# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayaahmed571/Flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — The Anatomy of Growing Content

The paper reports that growing content was longer and younger than declining content.
Growing pages averaged about 3,180 words and 184 days of age, compared with 2,311 words and 230 days for declining pages.

**My methodology question:**
Where does the label come from? The paper defines growing and declining content using impression trend direction, so I would want to confirm that this is an observed performance outcome measured over a later time window rather than a manually assigned rule.

**Validation question:**
Does the validation design support the strength of the claim? Since this is an observational comparison between growing and declining pages, the result shows an association but does not establish that increasing word count or reducing content age causes growth.

I would treat this as directional evidence about the portfolio rather than a causal claim.

### Finding 2 — The Content Performance Curve

The paper reports that content performance peaks around 61–90 days, declines after 270 days, and that some older pages show stronger performance after being refreshed.

**My methodology question:**
Where does the label or outcome come from? I would want to confirm that the health score and impression comparisons are measured outcomes from the reported performance windows, and understand exactly how the refreshed and non-refreshed groups were defined.

**Validation question:**
Does the validation design carry the claim? Because the study is observational, older pages that were refreshed may differ from older pages that were not refreshed in other important ways. Therefore, the comparison can show an association between refresh status and performance, but it does not by itself prove that refreshing caused the improvement.

I would interpret this as directional evidence supporting refresh prioritization, not as a causal estimate of the effect of refreshing.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

In Week 5, the Decision Tree achieved a precision of 0.579, compared with 0.509 for the baseline.

For this audit, I re-ran the model using a grouped split by client_id. This prevents content from the same client from appearing in both the training and test sets.

The grouped split is more honest for this dataset because content from the same client can share similar characteristics. The model is therefore evaluated on clients it did not see during training.

In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining_label"].value_counts())

from sklearn.model_selection import GroupShuffleSplit

features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = df[features].copy()
y = df["is_declining_label"]
groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print(
    "Clients in both train and test:",
    len(
        set(groups.iloc[train_idx])
        & set(groups.iloc[test_idx])
    )
)

is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Train shape: (23837, 6)
Test shape: (6163, 6)
Clients in both train and test: 0


In [13]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

grouped_precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

print("Week-5 / before:", 0.579)
print("Grouped / after:", round(grouped_precision, 3))

Week-5 / before: 0.579
Grouped / after: 0.547


### Before vs after

The Week-5 model achieved a measured precision of 0.579.

Under the grouped-by-client split, precision was 0.547.

The grouped result is lower than the original result by 0.032. This suggests that the original evaluation may have benefited from having content from the same clients represented across the split. The grouped result provides a more honest estimate of performance on unseen clients.

I treat this result as measured, decision-support evidence rather than evidence that the model will generalize to every client.

## 3. Leakage audit

I checked the final six features for target leakage and future-window leakage.

The target is `is_declining_label`, which is derived from `trend_direction`. Therefore, `trend_direction` and `trend_pct` are excluded from the feature set.

The final features are based on content age, update recency, historical impressions, average position, CTR, and word count. These are treated as information available at the decision point.

No label-derived columns or explicit future-window features are included in the final feature set.

In [14]:
final_features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

leakage_columns = [
    "is_declining_label",
    "trend_direction",
    "trend_pct"
]

print("Final features:")
print(final_features)

print("\nPotential leakage columns:")
print([col for col in leakage_columns if col in final_features])

Final features:
['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']

Potential leakage columns:
[]


In [15]:
for col in leakage_columns:
    print(f"{col}: {'IN FEATURES' if col in final_features else 'NOT IN FEATURES'}")

is_declining_label: NOT IN FEATURES
trend_direction: NOT IN FEATURES
trend_pct: NOT IN FEATURES


## 4. Claim rewrite

### Original claim

The Decision Tree model performs better than the baseline and can effectively identify declining content.

### Safer claim

On the evaluated grouped-by-client split, the Decision Tree achieved a measured precision of 0.547, compared with 0.509 for the Week-4 baseline. This is observed, directional evidence that the model may provide useful decision-support for prioritizing potentially declining content. The result does not establish that the model will generalize to all clients or that the selected features cause content to decline.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.